In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Folder containing this script (also the output folder). Jupyter/Colab has no __file__, so fall back to the current working directory.
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
OUT = HERE

def find_raw(name: str) -> Path:
    """Look for a raw file in order: parent folder -> this folder -> current working directory.
    Works both locally (raw data one level up) and on Colab (all files in one folder)."""
    for d in (HERE.parent, HERE, Path.cwd()):
        p = d / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Raw file not found: {name}")

# Study window: body-type classification changed in 2022-07 -> start from that month; latest data is 2026-07
STUDY_START = pd.Timestamp("2022-07-01")
STUDY_END   = pd.Timestamp("2026-07-01")

log_rows = []          # cleaning log
def log(item, result):
    log_rows.append({"check": item, "result": result})
    print(f"  {item}: {result}")

print("=" * 78)
print("BYD data cleaning")
print("=" * 78)
print("Script folder:", HERE)
print("Output folder:", OUT)
print()

BYD data cleaning
Script folder: C:\Users\56426\Desktop\ABD\BYD EV\cleaned_data
Output folder: C:\Users\56426\Desktop\ABD\BYD EV\cleaned_data



In [2]:
# 1. M03 monthly registrations -- analysis backbone (same cleaning definitions as teammates)
# =============================================================================
print("[1] Cleaning M03-Car_Regn_by_make.csv")

m03_path = find_raw("M03-Car_Regn_by_make.csv")
source_hash = hashlib.sha256(m03_path.read_bytes()).hexdigest()

# Read everything as strings so pandas' automatic NA/type inference does not affect the checks
raw = pd.read_csv(m03_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()

required_columns = ["month", "make", "importer_type", "fuel_type", "vehicle_type", "number"]
missing_columns = [c for c in required_columns if c not in raw.columns]
if missing_columns:
    raise ValueError(f"M03 missing columns: {missing_columns}")

df = raw[required_columns].copy()
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print(f"  Raw rows: {len(raw)}  columns: {raw.columns.tolist()}")
log("M03 raw rows", len(raw))
log("M03 source file SHA256", source_hash[:16] + "...")

# --- 1.1 Parse month ---
parsed_month = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
bad_month_count = int(parsed_month.isna().sum())
if bad_month_count:
    raise ValueError("Unparseable months found; please review.")
df["month"] = parsed_month
log("M03 rows with unparseable month", bad_month_count)

# --- 1.2 Restrict to study window ---
df = df.loc[df["month"].between(STUDY_START, STUDY_END)].copy()
if df.empty:
    raise ValueError("No records within the study window.")
log("M03 study window", "2022-07 to 2026-07")
log("M03 rows in window", len(df))

# --- 1.3 Required analysis fields must not be blank (month / make / fuel_type / vehicle_type) ---
analysis_required_key = ["month", "make", "fuel_type", "vehicle_type"]
missing_key_rows = df[analysis_required_key].eq("").any(axis=1) if "vehicle_type" in df else None
# month is already datetime, so only check the text fields here
missing_key_rows = (df["make"].eq("") | df["fuel_type"].eq("") | df["vehicle_type"].eq(""))
missing_key_count = int(missing_key_rows.sum())
if missing_key_count:
    raise ValueError("Required analysis fields still have missing values; please review.")
log("M03 rows missing key fields", missing_key_count)

# --- 1.4 Counts: blank -> 0, must be non-negative integers ---
blank_number_count = int(df["number"].eq("").sum())
parsed_number = pd.to_numeric(df["number"].replace("", "0"), errors="coerce")
bad_number_rows = parsed_number.isna() | parsed_number.lt(0) | parsed_number.mod(1).ne(0)
bad_number_count = int(bad_number_rows.sum())
if bad_number_count:
    raise ValueError("Registrations contain unparseable, negative or non-integer values.")
df["registrations"] = parsed_number.astype("int64")
log("M03 blank counts treated as 0", blank_number_count)     # since 2018-07 LTA uses blanks to mean zero
log("M03 rows with invalid counts", bad_number_count)

# --- 1.5 Check raw duplicates (by month/make/importer/fuel/body) ---
original_key = ["month", "make", "importer_type", "fuel_type", "vehicle_type"]
dup_rows = df.duplicated(subset=original_key, keep=False)
duplicate_row_count = int(dup_rows.sum())
if duplicate_row_count:
    raise ValueError("Duplicate records found; confirm the cause first.")
log("M03 raw duplicate rows", duplicate_row_count)

# --- 1.6 Importer check: unspecified (blank) and specified importer rows with positive sales must not coexist in the same group ---
importer_missing = df["importer_type"].eq("")
unspecified = df.loc[importer_missing & df["registrations"].gt(0), analysis_required_key].drop_duplicates()
specified   = df.loc[~importer_missing & df["registrations"].gt(0), analysis_required_key].drop_duplicates()
overlap = unspecified.merge(specified, on=analysis_required_key, how="inner")
overlap_count = len(overlap)
if overlap_count:
    raise ValueError("Unspecified and specified importer records coexist in the same group; please review.")
log("M03 importer overlap groups", overlap_count)

# --- 1.7 Standardise body types ---
body_mapping = {
    "Sports Utility Vehicle": "SUV",
    "Sedan":                  "Sedan",
    "Multi-purpose Vehicle":  "MPV",
    "Hatchback":              "Hatchback",
    "Station-wagon":          "Station Wagon",
    "Coupe/ Convertible":     "Coupe/Convertible",
}
unknown_body_types = sorted(set(df["vehicle_type"]) - set(body_mapping))
if unknown_body_types:
    raise ValueError(f"Unmapped body types: {unknown_body_types}")
df["body"] = df["vehicle_type"].map(body_mapping)
log("M03 body type mapping", "SUV/Sedan/MPV/Hatchback/Station Wagon/Coupe-Convertible")

# --- 1.8 Aggregate to analysis grain: month x make x fuel x body (importer types summed out) ---
fact_key = ["month", "make", "fuel_type", "body"]
total_before = int(df["registrations"].sum())
registration_fact = (df.groupby(fact_key, as_index=False)["registrations"]
                       .sum()
                       .sort_values(fact_key)
                       .reset_index(drop=True))
total_after = int(registration_fact["registrations"].sum())

fact_duplicate_count = int(registration_fact.duplicated(fact_key).sum())
target_bodies = ["SUV", "Sedan", "MPV"]
month_counts = (registration_fact.loc[registration_fact["body"].isin(target_bodies)]
                .groupby("body")["month"].nunique().reindex(target_bodies, fill_value=0))

if fact_duplicate_count: raise ValueError("Duplicates at final grain.")
if total_before != total_after: raise ValueError("Totals differ before and after aggregation.")
if not registration_fact["registrations"].ge(0).all(): raise ValueError("Negative registrations found.")
if not month_counts.eq(49).all(): raise ValueError("Month coverage for target body types is incomplete.")

log("M03 total registrations before aggregation", total_before)
log("M03 total registrations after aggregation", total_after)
log("M03 final table rows", len(registration_fact))
log("M03 duplicate keys in final table", fact_duplicate_count)
for b in target_bodies:
    log(f"M03 {b} months covered", int(month_counts[b]))

# Export month as YYYY-MM text for easy sorting in Excel
export_fact = registration_fact.copy()
export_fact["month"] = export_fact["month"].dt.strftime("%Y-%m")
export_fact.to_csv(OUT / "registration_fact.csv", index=False, encoding="utf-8-sig")
print(f"  -> registration_fact.csv  rows {len(export_fact)}")
print()

[1] Cleaning M03-Car_Regn_by_make.csv
  Raw rows: 61359  columns: ['month', 'make', 'importer_type', 'fuel_type', 'vehicle_type', 'number']
  M03 raw rows: 61359
  M03 source file SHA256: dae4a01794d45a5a...
  M03 rows with unparseable month: 0
  M03 study window: 2022-07 to 2026-07
  M03 rows in window: 33444
  M03 rows missing key fields: 0
  M03 blank counts treated as 0: 25803
  M03 rows with invalid counts: 0
  M03 raw duplicate rows: 0
  M03 importer overlap groups: 0
  M03 body type mapping: SUV/Sedan/MPV/Hatchback/Station Wagon/Coupe-Convertible
  M03 total registrations before aggregation: 172394
  M03 total registrations after aggregation: 172394
  M03 final table rows: 26898
  M03 duplicate keys in final table: 0
  M03 SUV months covered: 49
  M03 Sedan months covered: 49
  M03 MPV months covered: 49
  -> registration_fact.csv  rows 26898



In [3]:
# 2. COE (Certificate of Entitlement) -- convert to numeric (raw values contain thousands separators)
# =============================================================================
print("[2] Cleaning COEBiddingResultsPrices.csv")
coe_path = find_raw("COEBiddingResultsPrices.csv")
coe = pd.read_csv(coe_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
coe.columns = coe.columns.str.strip()
for col in ["month", "bidding_no", "vehicle_class", "quota", "bids_success", "bids_received", "premium"]:
    coe[col] = coe[col].fillna("").astype(str).str.strip()

coe["month"] = pd.to_datetime(coe["month"], format="%Y-%m", errors="coerce")
if coe["month"].isna().any():
    raise ValueError("COE month has invalid values.")

comma_cols = ["quota", "bids_success", "bids_received", "premium"]
comma_counts = {c: int(coe[c].str.contains(",").sum()) for c in comma_cols}
for c in comma_cols:                       # strip thousands separators, then convert to numbers
    coe[c] = pd.to_numeric(coe[c].str.replace(",", "", regex=False), errors="coerce")
if coe[comma_cols].isna().any().any():
    raise ValueError("COE numeric columns contain unparseable values.")
coe["bidding_no"] = coe["bidding_no"].astype(int)

for c, n in comma_counts.items():
    log(f"COE {c} rows with thousands separators", n)
log("COE rows", len(coe))
log("COE month range", f"{coe['month'].min():%Y-%m} to {coe['month'].max():%Y-%m}")
coe.to_csv(OUT / "coe_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> coe_clean.csv  rows {len(coe)}")
print()

[2] Cleaning COEBiddingResultsPrices.csv
  COE quota rows with thousands separators: 0
  COE bids_success rows with thousands separators: 41
  COE bids_received rows with thousands separators: 109
  COE premium rows with thousands separators: 0
  COE rows: 1975
  COE month range: 2010-01 to 2026-09
  -> coe_clean.csv  rows 1975



In [4]:
# 3. M09 vehicle population
# =============================================================================
print("[3] Cleaning M09-Vehs_by_Fuel_Type.csv")
m09_path = find_raw("M09-Vehs_by_Fuel_Type.csv")
m09 = pd.read_csv(m09_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
m09.columns = m09.columns.str.strip()
for col in m09.columns:
    m09[col] = m09[col].fillna("").astype(str).str.strip()
m09["month"] = pd.to_datetime(m09["month"], format="%Y-%m", errors="coerce")
if m09["month"].isna().any():
    raise ValueError("M09 month has invalid values.")
# In the source file "-" is a placeholder (missing); treat as 0 and log the count
dash_count = int(m09["number"].eq("-").sum())
m09["number"] = m09["number"].replace({"": "0", "-": "0"})
m09["number"] = pd.to_numeric(m09["number"].str.replace(",", "", regex=False), errors="coerce")
if m09["number"].isna().any():
    raise ValueError("M09 number contains unparseable values.")
m09["number"] = m09["number"].astype("int64")
log("M09 '-' placeholders treated as 0", dash_count)
m09 = m09[["month", "category", "type", "number"]].sort_values(["month", "category", "type"]).reset_index(drop=True)
log("M09 rows", len(m09))
log("M09 month range", f"{m09['month'].min():%Y-%m} to {m09['month'].max():%Y-%m}")
m09.to_csv(OUT / "fleet_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> fleet_clean.csv  rows {len(m09)}")
print()

[3] Cleaning M09-Vehs_by_Fuel_Type.csv
  M09 '-' placeholders treated as 0: 5
  M09 rows: 3611
  M09 month range: 2016-01 to 2026-07
  -> fleet_clean.csv  rows 3611



In [5]:
# 4. MVP02-2 annual new car registrations
# =============================================================================
print("[4] Cleaning MVP02-2_New_Cars_by_make_type.csv")
mvp02_path = find_raw("MVP02-2_New_Cars_by_make_type.csv")
mvp02 = pd.read_csv(mvp02_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
mvp02.columns = mvp02.columns.str.strip()
for col in mvp02.columns:
    mvp02[col] = mvp02[col].fillna("").astype(str).str.strip()
mvp02["year"] = pd.to_numeric(mvp02["year"], errors="coerce").astype("Int64")
mvp02["number"] = mvp02["number"].replace({"": "0", "-": "0"})
mvp02["number"] = pd.to_numeric(mvp02["number"].str.replace(",", "", regex=False), errors="coerce").fillna(0).astype("int64")
mvp02 = mvp02[["year", "make", "importer_type", "fuel", "type", "number"]].sort_values(["year", "make", "fuel", "type"]).reset_index(drop=True)
log("MVP02-2 rows", len(mvp02))
log("MVP02-2 year range", f"{int(mvp02['year'].min())}-{int(mvp02['year'].max())}")
mvp02.to_csv(OUT / "mvp02_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> mvp02_clean.csv  rows {len(mvp02)}")
print()

[4] Cleaning MVP02-2_New_Cars_by_make_type.csv
  MVP02-2 rows: 6896
  MVP02-2 year range: 2015-2025
  -> mvp02_clean.csv  rows 6896



In [6]:
# 5. MVP01-6 annual registrations (by make)
# =============================================================================
print("[5] Cleaning MVP01-6_Cars_by_make.csv")
mvp01_path = find_raw("MVP01-6_Cars_by_make.csv")
mvp01 = pd.read_csv(mvp01_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
mvp01.columns = mvp01.columns.str.strip()
for col in mvp01.columns:
    mvp01[col] = mvp01[col].fillna("").astype(str).str.strip()
mvp01["year"] = pd.to_numeric(mvp01["year"], errors="coerce").astype("Int64")
mvp01["number"] = mvp01["number"].replace({"": "0", "-": "0"})
mvp01["number"] = pd.to_numeric(mvp01["number"].str.replace(",", "", regex=False), errors="coerce").fillna(0).astype("int64")
mvp01 = mvp01[["year", "make", "fuel_type", "number"]].sort_values(["year", "make"]).reset_index(drop=True)
log("MVP01-6 rows", len(mvp01))
log("MVP01-6 year range", f"{int(mvp01['year'].min())}-{int(mvp01['year'].max())}")
mvp01.to_csv(OUT / "mvp01_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> mvp01_clean.csv  rows {len(mvp01)}")
print()

[5] Cleaning MVP01-6_Cars_by_make.csv
  MVP01-6 rows: 2198
  MVP01-6 year range: 2005-2025
  -> mvp01_clean.csv  rows 2198



In [7]:
# 6. EV charger registry (standardise column names + parse dates/coordinates + de-duplicate)
# =============================================================================
print("[6] Cleaning Electric_Vehicle_Charging_Points_Jun 2026.csv")
ch_path = find_raw("Electric_Vehicle_Charging_Points_Jun 2026.csv")
ch = pd.read_csv(ch_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
ch.columns = ch.columns.str.strip()

rename = {
    "EV Charger Registration Code": "charger_code",
    "operator": "operator",
    "No. of Charging Outlets": "outlets",
    "evCpId": "cp_id",
    "plugType": "plug_type",
    "charging Speed": "charging_speed_kw",
    "PostalCode": "postal_code",
    "Block/House No": "block_no",
    "Street Name": "street_name",
    "Building Name": "building_name",
    "Floor No": "floor_no",
    "Lot No": "lot_no",
    "Is the charger publicly accessible?": "public_accessible",
    "longitude": "longitude",
    "latitude": "latitude",
    "Registration Date": "registration_date",
    "Type of Parking Lot": "parking_lot_type",
}
ch = ch.rename(columns=rename)
for col in ch.columns:
    ch[col] = ch[col].fillna("").astype(str).str.strip()

ch["outlets"] = pd.to_numeric(ch["outlets"].replace("", "0"), errors="coerce").fillna(0).astype(int)
ch["charging_speed_kw"] = pd.to_numeric(ch["charging_speed_kw"], errors="coerce")
ch["longitude"] = pd.to_numeric(ch["longitude"], errors="coerce")
ch["latitude"] = pd.to_numeric(ch["latitude"], errors="coerce")
ch["registration_date"] = pd.to_datetime(ch["registration_date"], errors="coerce", dayfirst=False)

before = len(ch)
ch = ch.drop_duplicates(subset=["cp_id"]).reset_index(drop=True)
log("Chargers raw rows", before)
log("Chargers rows after de-duplication", len(ch))
log("Chargers total outlets", int(ch["outlets"].sum()))
log("Chargers registration date range", f"{ch['registration_date'].min():%Y-%m-%d} to {ch['registration_date'].max():%Y-%m-%d}")
ch.to_csv(OUT / "chargers_clean.csv", index=False, encoding="utf-8-sig")
print(f"  -> chargers_clean.csv  rows {len(ch)}")
print()

[6] Cleaning Electric_Vehicle_Charging_Points_Jun 2026.csv
  Chargers raw rows: 11353
  Chargers rows after de-duplication: 11353
  Chargers total outlets: 16935
  Chargers registration date range: 2024-01-03 to 2026-12-04
  -> chargers_clean.csv  rows 11353

